# Final Project

## Project Description

This project is a productivity tool that combines a Pomodoro timer with an interactive Snake game built using the Pygame library. The Pomodoro technique is a time management method that splits work into focused intervals, typically 25 minutes followed by short breaks, helping users stay productive by making the tasks more approachable. The program allows users to set a custom study timer that counts down and notifies them when it is time to take a break. Between study sessions, the user is presented with a menu offering five options: restart the same study timer, reset new study and break durations, start a break countdown, launch the Snake game for the break period, or quit the program entirely. If the user chooses to play the Snake game, a window will open, giving the user a fun and engaging way to relax during their break. Crucially, the game is timed by the break clock and will automatically close when the break period expires, returning the user to the terminal menu so they can get back to their study session. (Note that when the user presses ESC or clicks Return to Menu, the game window may briefly linger before fully closing.)

As students ourselves, we wanted to create something that was both practical and enjoyable. Staying focused during long study sessions can be challenging, and we often find ourselves getting distracted or losing track of time during breaks. This project was inspired by the idea of having a built-in, guilt-free way to recharge between sessions without breaking focus. By combining the Pomodoro study technique with a classic game, we hoped to build a tool that could make studying feel a little less overwhelming and a little more rewarding. 
    

    
#### References (*if applicable*)

Pomodoro clock reference: https://docs.python.org/3/library/time.html

User input reference: https://www.geeksforgeeks.org/python/taking-input-in-python/

The Snake Game component was developed with the assistance of Claude AI

## Team Contributions
* Team Member 1: Tin Vo
* Team Member 2: Dusan Djordjevic 
* Team Member 3: Selina Yang 

    For this project, our team collaborated closely through weekly online meetings to build a Pomodoro timer with an integrated Snake game. Tien developed the core Pomodoro countdown functionality, while Selina handled user input and error catching. Dusan built the interactive Snake game using Pygame, giving users a fun activity to enjoy during their breaks. All three of us worked together to integrate the components into a cohesive final program. To wrapped up final project, we all tested the code on our own devices to make sure everything worked as expected.

<hr>

## Setup

In [ ]:
import time
import sys as sy
import random as rd
import pygame as pg

## Main Code

In [ ]:

def show_break_timer(break_seconds):
    """Counts down the break time in the terminal. Uses the break_seconds parameter to count down until it equals 0. """
    seconds = break_seconds
    while seconds > 0: 
        mins = seconds // 60
        secs = seconds % 60
        print(f"{mins:02d}:{secs:02d} left", end="\r")  # Claude assisted - '\r' moves the cursor back to the start of the line so the timer counts down in the same line instead of printing a new line each second.
        time.sleep(1)
        seconds -= 1
    print("\nBreak over! Back to work.")


# Each cell is 32x32px, GRID_W and GRID_H are the width and height of the playable area in number of cells.
# HUD_HEIGHT is the height of the bar at the top.
# FPS controls how fast the snake moves.
def play_snake_game(break_seconds):
    """Runs the snake game for the inputted break time. Uses the break_seconds parameter to exit to the menu when the timer runs out or the user presses ESC."""
    pg.init()
    CELL_SIZE = 32
    GRID_W = 24
    GRID_H = 24
    HUD_HEIGHT = 24                        
    width = GRID_W * CELL_SIZE
    height = GRID_H * CELL_SIZE + HUD_HEIGHT  
    FPS = 12

    # colors
    BLACK = (0, 0, 0)
    WHITE = (245, 245, 245)
    YELLOW = (255, 220, 0)
    DARK_YELLOW = (180, 140, 0)
    RED = (220, 40, 40)
    GRAY = (35, 35, 35)
    DARK_GRAY = (20, 20, 20)
    LIGHT_GRAY = (180, 180, 180)
    GREEN = (50, 200, 100)
    BTN_COLOR = (50, 50, 50)
    BTN_HOVER = (80, 80, 80)

    def random_food_position(snake):
        """Picks a random position for the food to spawn. Uses snake parameter to avoid spawning the food on the snake."""
        # Keeps trying spots until it finds one not on the snake.
        while True:
            x = rd.randint(0, GRID_W - 1)
            y = rd.randint(0, GRID_H - 1)
            pos = (x, y)
            if pos not in snake:
                return pos

    # This function gives python instructions on how to draw cells in the game window.
    # Shifts the y position down by HUD_HEIGHT so cells draw below the HUD and within the playfield.
    # rect converts grid position to pixel coordinates, while accounting for the displacement by unplayable area (HUD_HEIGHT)
    # Utilized to draw the snake's head, body, and food.
    def draw_cell(pos, color):
        """Draws a single cell at the chosen position. Uses pos and color parameters to determine where to draw the cell and what color to make it. The y value is shifted down by HUD_HEIGHT so cells appear inside the playfield and not behind the HUD."""
        rect = pg.Rect(pos[0] * CELL_SIZE, pos[1] * CELL_SIZE + HUD_HEIGHT, CELL_SIZE, CELL_SIZE)  
        pg.draw.rect(screen, color, rect)
        pg.draw.rect(screen, GRAY, rect, 1)

    # This function defines the process for drawing menu buttons once the game is over.
    def draw_button(text, rect, mouse_pos):
        """Draws a clickable button and highlights it when the mouse is hovering over it. Centers the label text inside the button area."""
        color = BTN_HOVER if rect.collidepoint(mouse_pos) else BTN_COLOR  # Claude assisted - one line if/else: uses BTN_HOVER if the mouse is over the button, otherwise uses BTN_COLOR.
        pg.draw.rect(screen, color, rect, border_radius=6)
        pg.draw.rect(screen, LIGHT_GRAY, rect, 1, border_radius=6)
        label = font.render(text, True, WHITE)
        screen.blit(label, (rect.centerx - label.get_width() // 2,
                            rect.centery - label.get_height() // 2))

    screen = pg.display.set_mode((width, height))
    pg.display.set_caption('Snake - Break Time')
    clock = pg.time.Clock()
    font = pg.font.SysFont('arial', 22)
    font_big = pg.font.SysFont('arial', 36, bold=True)

    # This function sets the game back to its starting state.
    # Returns the same starting snake (always 3 cells long), same initial direction, and same pending direction.
    # Also generates a new random food position, resets the score to 0, and sets game_over to False.
    def reset_snake():
        """Puts the game back to its starting state. Returns a new snake, direction, next_dir, food position, score of 0, and lost set to False."""
        s = [(GRID_W // 2, GRID_H // 2),
             (GRID_W // 2 - 1, GRID_H // 2),
             (GRID_W // 2 - 2, GRID_H // 2)]
        return s, (1, 0), (1, 0), random_food_position(s), 0, False

    snake, direction, next_dir, food, score, lost = reset_snake()  # Claude assisted - unpacks all 6 return values from reset_snake() into individual variables in one line.

    break_end_time = time.time() + break_seconds  # Claude assisted - records the exact system clock time when the break should end, so the countdown is always accurate regardless of game performance.

    # Defines the area where buttons will be drawn in the game over screen.
    btn_restart = pg.Rect(width // 2 - 220, height // 2 + 30, 180, 44)
    btn_menu = pg.Rect(width // 2 + 40, height // 2 + 30, 180, 44)
    btn_quit = pg.Rect(width // 2 - 70, height // 2 + 90, 140, 44)

    while True:
        time_left = break_end_time - time.time()  # Claude assisted - calculates how many seconds are left by subtracting the current system time from the recorded end time.
        mouse_pos = pg.mouse.get_pos()

        # If time_left <= 0 the game ends.
        if time_left <= 0:
            pg.quit()
            return

        # Checks for keyboard and mouse actions each frame.
        # QUIT closes the program, ESCAPE returns to the menu.
        for event in pg.event.get():
            if event.type == pg.QUIT:
                pg.quit()
                sy.exit()
            if event.type == pg.KEYDOWN:
                if event.key == pg.K_ESCAPE:
                    pg.quit()
                    return
                if lost:
                    if event.key == pg.K_r:
                        snake, direction, next_dir, food, score, lost = reset_snake()

                
                # arrow/WASD movement, blocks reversing into yourself.
                else:
                    if event.key in (pg.K_LEFT, pg.K_a) and direction != (1, 0):
                        next_dir = (-1, 0)
                    elif event.key in (pg.K_RIGHT, pg.K_d) and direction != (-1, 0):
                        next_dir = (1, 0)
                    elif event.key in (pg.K_UP, pg.K_w) and direction != (0, 1): 
                        next_dir = (0, -1)
                    elif event.key in (pg.K_DOWN, pg.K_s) and direction != (0, -1):
                        next_dir = (0, 1)
            if event.type == pg.MOUSEBUTTONDOWN:
                if lost:
                    if btn_restart.collidepoint(event.pos):
                        snake, direction, next_dir, food, score, lost = reset_snake()
                    elif btn_menu.collidepoint(event.pos):
                        pg.quit()
                        return
                    elif btn_quit.collidepoint(event.pos):
                        pg.quit()
                        sy.exit()

        # If the game is not over allows the snake to continue moving.
        if not lost:
            direction = next_dir
            new_head = (snake[0][0] + direction[0], snake[0][1] + direction[1])
            if (new_head[0] < 0 or new_head[0] >= GRID_W or
                    new_head[1] < 0 or new_head[1] >= GRID_H):
                lost = True
            elif new_head in snake:
                lost = True
            else:
                snake.insert(0, new_head)  # Claude assisted - insert(0, ...) adds the new head to the front of the snake list rather than the end.
                if new_head == food:
                    score += 1
                    food = random_food_position(snake)
                else:
                    snake.pop()  # Claude assisted - pop() removes the last item in the list (the tail), so the snake moves forward without growing.

        # Draws the playfield, snake, and food.
        screen.fill(BLACK)
        draw_cell(food, RED)
        draw_cell(snake[0], DARK_YELLOW)
        for part in snake[1:]:  # Claude assisted - [1:] is list slicing, meaning loop over every segment except the first item (the head).
            draw_cell(part, YELLOW)

        # Draws the HUD and components.
        pg.draw.rect(screen, DARK_GRAY, (0, 0, width, HUD_HEIGHT))
        secs_left = max(0, int(time_left))  # Claude assisted - max() prevents the timer from displaying a negative number if time_left drops below 0.
        bm, bs = secs_left // 60, secs_left % 60
        score_surf = font.render(f'Score: {score}', True, WHITE)  # Claude assisted - uses pygame's font system to turn the score text into a drawable surface.
        break_surf = font.render(f'Break: {bm:02d}:{bs:02d}', True, YELLOW)  # Claude assisted - uses pygame's font system to turn the timer text into a drawable surface; :02d formats each number as 2 digits with a leading zero if needed (e.g. 5 becomes 05).
        esc_surf = font.render('ESC = menu', True, LIGHT_GRAY)  # Claude assisted - uses pygame's font system to turn the ESC hint text into a drawable surface.

        # Draws the HUD text for the Score, Break Time, and a reminder that 'ESC' quits the game.
        screen.blit(score_surf, (10, HUD_HEIGHT // 2 - score_surf.get_height() // 2))
        screen.blit(break_surf, (width // 2 - break_surf.get_width() // 2, HUD_HEIGHT // 2 - break_surf.get_height() // 2))
        screen.blit(esc_surf, (width - esc_surf.get_width() - 10, HUD_HEIGHT // 2 - esc_surf.get_height() // 2))

        # When the game is lost, this darkens the screen and displays the game over message with the menu, restart, and quit buttons.
        if lost:
            overlay = pg.Surface((width, height), pg.SRCALPHA)  # Claude assisted - creates a separate transparent surface the same size as the window that can be drawn on top of the game.
            overlay.fill((0, 0, 0, 160))  # Claude assisted - fills the overlay with black at 160/255 opacity, creating a semi-transparent darkening effect over the game.
            screen.blit(overlay, (0, 0))
            go_surf = font_big.render('Game Over!', True, RED)
            screen.blit(go_surf, (width // 2 - go_surf.get_width() // 2, height // 2 - 50))
            draw_button('R  Restart', btn_restart, mouse_pos)
            draw_button('Return to Menu', btn_menu, mouse_pos)
            draw_button('Quit', btn_quit, mouse_pos)

        pg.display.flip()  # Claude assisted - updates the entire screen each frame so the player sees the latest drawn state; without this nothing would appear on screen.
        clock.tick(FPS)


def pomodoro(minutes):
    """Takes a number of minutes and counts it down in the terminal. Prints a break reminder when it hits zero."""
    seconds = int(minutes * 60)
    while seconds > 0:
        mins = int(seconds // 60)
        secs = int(seconds % 60)
        print(f"{mins:02d}:{secs:02d} left", end="\r")  # Claude assisted - '\r' moves the cursor back to the start of the line so the timer overwrites itself instead of printing a new line each second.
        time.sleep(1)
        seconds -= 1
    print("\nTake a break!")


# Ask input for study time and break time.
while True:
    while True:
        try:
            study_minutes = float(input("Enter study time (minutes): "))
            if study_minutes <= 0:                                                #Error catching for time less than zero
                print('Invalid input, please enter a number greater than 0')
                continue
            break
        except ValueError:                                                        #Error catching for ValueError
            print("Invalid input. Please enter a number (e.g., 25)")

    # Keeps asking for the break time until it gets a number.
    while True:
        try:
            break_minutes = float(input("Enter break time (minutes): "))
            if break_minutes <= 0:
                print('Invalid input, please enter a number greater than 0')
                continue
            break
        except ValueError:
            print("Invalid input. Please enter a number (e.g., 5)")

    break_seconds = int(break_minutes * 60)  # Converts the time in minutes into seconds.

    # After the study timer ends, this presents the user with options to restart, set a new timer, start the break timer, play snake, or quit.
    while True:
        pomodoro(study_minutes)
        print("\nWhat would you like to do?")
        choice = input('1. Restart timer\n2. Set a new timer\n3. Start break timer\n4. Play Snake\n5. Quit\n> ')
        if choice == '1':        #Restarts the while loop
            continue
        elif choice == '2':      #Exits the while loop and returns to the prompt for new study and break times
            break
        elif choice == '3':      #Runs the break countdown
            show_break_timer(break_seconds)
            continue
        elif choice == '4':      #Open Snake game window for the duration of the break
            play_snake_game(break_seconds)
            print('Break over! Back to work.')
            continue
        elif choice == '5':      #Exit program entirely
            print('Goodbye!')
            sy.exit()
        else:
            print('Invalid choice, timer will restart.')